In [1]:
import pandas as pd
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
from google.colab import drive

# Mount drive if not done
drive.mount('/content/drive', force_remount=True)

# Path to your folder
drive_path = '/content/drive/MyDrive/MIMIC_III_REAL/'

print("Reading datasets...")
adm = pd.read_csv(drive_path + 'ADMISSIONS.csv.gz', compression='gzip')
pat = pd.read_csv(drive_path + 'PATIENTS.csv.gz', compression='gzip')
icu = pd.read_csv(drive_path + 'ICUSTAYS.csv.gz', compression='gzip')
diag = pd.read_csv(drive_path + 'DIAGNOSES_ICD.csv.gz', compression='gzip')
d_diag = pd.read_csv(drive_path + 'D_ICD_DIAGNOSES.csv.gz', compression='gzip')

# 1. Standardize column names to lowercase
for df in [adm, pat, diag, d_diag, icu]:
    df.columns = df.columns.str.lower()

# 2. FIX MIMIC-III SPECIFIC COLUMN NAMES (Map icd9_code -> icd_code)
if 'icd9_code' in diag.columns:
    diag = diag.rename(columns={'icd9_code': 'icd_code'})
if 'icd9_code' in d_diag.columns:
    d_diag = d_diag.rename(columns={'icd9_code': 'icd_code'})

print("All base files loaded and schemas standardized successfully!")
print("Proceeding with cardiac cohort extraction...")

# 3. Filter for cardiac diagnoses
cardiac_keywords = 'heart|cardiac|coronary|infarction|failure|arrest|tachycardia|arrhythmia'
cardiac_codes = d_diag[d_diag['long_title'].str.contains(cardiac_keywords, case=False, na=False)]
cardiac_diag = diag[diag['icd_code'].isin(cardiac_codes['icd_code'])]

# 4. Extract cohort entries
mega = pd.merge(adm, pat, on='subject_id', how='inner')
mega = mega[mega['hadm_id'].isin(cardiac_diag['hadm_id'])].copy()

# Add clinical descriptions
pri_diag = pd.merge(cardiac_diag, d_diag, on='icd_code', how='left')
pri_diag = pri_diag.drop_duplicates(subset=['hadm_id'])[['hadm_id', 'long_title']]
mega = pd.merge(mega, pri_diag, on='hadm_id', how='left')

# 5. Process ICU Stays
cardiac_icu = pd.merge(mega, icu, on=['subject_id', 'hadm_id'], how='inner')

cardiac_icu['intime'] = pd.to_datetime(cardiac_icu['intime'])
cardiac_icu['outtime'] = pd.to_datetime(cardiac_icu['outtime'])
cardiac_icu['total_icu_hours'] = ((cardiac_icu['outtime'] - cardiac_icu['intime']).dt.total_seconds() / 3600).round(2)
cardiac_icu['unit_path'] = cardiac_icu['first_careunit'] + " -> " + cardiac_icu['last_careunit']

# 6. Final Feature Matrix Configuration
cardiac_icu['death_event'] = cardiac_icu['hospital_expire_flag'].fillna(0).astype(int)

cardiac_icu['admittime'] = pd.to_datetime(cardiac_icu['admittime'])
cardiac_icu['dob'] = pd.to_datetime(cardiac_icu['dob'])
cardiac_icu['anchor_age'] = (cardiac_icu['admittime'].dt.year - cardiac_icu['dob'].dt.year)
cardiac_icu.loc[cardiac_icu['anchor_age'] > 89, 'anchor_age'] = 90

cardiac_icu['dischtime'] = pd.to_datetime(cardiac_icu['dischtime'])
cardiac_icu['total_los_days'] = ((cardiac_icu['dischtime'] - cardiac_icu['admittime']).dt.total_seconds() / 86400).round(2)
cardiac_icu['stay_window'] = cardiac_icu['admittime'].dt.strftime('%Y-%m-%d') + " to " + cardiac_icu['dischtime'].dt.strftime('%Y-%m-%d')

# Filter for tracking keys and target indicators
final_columns = [
    'subject_id', 'hadm_id', 'stay_id', 'gender', 'anchor_age',
    'long_title', 'total_los_days', 'unit_path', 'total_icu_hours',
    'death_event', 'stay_window'
]

df_final = cardiac_icu[[c for c in final_columns if c in cardiac_icu.columns]].copy()

# Output the processed matrix to Excel
output_file = drive_path + 'CARDIAC_MIMIC_III_EXPANDED.xlsx'
df_final.to_excel(output_file, index=False)
print(f"\nSUCCESS: Created model-ready dataset with {len(df_final)} rows.")

Mounted at /content/drive
Reading datasets...
All base files loaded and schemas standardized successfully!
Proceeding with cardiac cohort extraction...

SUCCESS: Created model-ready dataset with 39022 rows.


In [ ]:
import pandas as pd
import numpy as np
import os
from google.colab import drive

# 1. Mount and Setup Paths
drive.mount('/content/drive')
base_path = '/content/drive/MyDrive/MIMIC_III_REAL/'

print("Step 1: Loading core administrative and mapping files...")
adm = pd.read_csv(base_path + 'ADMISSIONS.csv.gz', compression='gzip')
pat = pd.read_csv(base_path + 'PATIENTS.csv.gz', compression='gzip')
diag = pd.read_csv(base_path + 'DIAGNOSES_ICD.csv.gz', compression='gzip')
d_diag = pd.read_csv(base_path + 'D_ICD_DIAGNOSES.csv.gz', compression='gzip')
proc = pd.read_csv(base_path + 'PROCEDURES_ICD.csv.gz', compression='gzip')
d_proc = pd.read_csv(base_path + 'D_ICD_PROCEDURES.csv.gz', compression='gzip')

# Convert all headers to lowercase immediately
for data_frame in [adm, pat, diag, d_diag, proc, d_proc]:
    data_frame.columns = data_frame.columns.str.lower()

# Fix MIMIC-III specific diagnosis mapping column names
for df_code in [diag, d_diag]:
    if 'icd9_code' in df_code.columns:
        df_code.rename(columns={'icd9_code': 'icd_code'}, inplace=True)

# Fix MIMIC-III specific procedure mapping column names
if 'icd9_code' in proc.columns:
    proc.rename(columns={'icd9_code': 'icd_code'}, inplace=True)
if 'icd9_code' in d_proc.columns:
    d_proc.rename(columns={'icd9_code': 'icd_code'}, inplace=True)

# DYNAMIC FIX FOR THE KEYERROR: Check how MIMIC-III named the procedure text column
if 'short_title' in d_proc.columns:
    d_proc.rename(columns={'short_title': 'long_title'}, inplace=True)
elif 'description' in d_proc.columns:
    d_proc.rename(columns={'description': 'long_title'}, inplace=True)

print("Step 2: Isolating the cardiac cohort admissions...")
cardiac_keywords = 'heart|cardiac|coronary|infarction|failure|arrest|tachycardia|arrhythmia'
cardiac_diag_codes = d_diag[d_diag['long_title'].str.contains(cardiac_keywords, case=False, na=False)]
cardiac_admissions_ids = diag[diag['icd_code'].isin(cardiac_diag_codes['icd_code'])]['hadm_id'].unique()

# Filter admissions and patients down to target cardiac group
adm_filtered = adm[adm['hadm_id'].isin(cardiac_admissions_ids)].copy()
df_master = pd.merge(adm_filtered, pat, on='subject_id', how='inner')

print("Step 3: Calculating clinical baselines...")
df_master['admittime'] = pd.to_datetime(df_master['admittime'])
df_master['dob'] = pd.to_datetime(df_master['dob'])
df_master['anchor_age'] = df_master['admittime'].dt.year - df_master['dob'].dt.year
df_master.loc[df_master['anchor_age'] > 89, 'anchor_age'] = 90  # Masking elderly outliers

df_master['is_deceased'] = df_master['hospital_expire_flag'].fillna(0).astype(int)

# Map diagnosis long title to master
diag_with_title = pd.merge(diag, d_diag, on='icd_code', how='left')
primary_diagnoses = diag_with_title[diag_with_title['seq_num'] == 1][['hadm_id', 'long_title', 'icd_code']]
df_master = pd.merge(df_master, primary_diagnoses, on='hadm_id', how='left')

print("Step 4: Extracting discharge summaries from NOTEVENTS (Processing in chunks)...")
notes_chunks = pd.read_csv(base_path + 'NOTEEVENTS.csv.gz', compression='gzip', chunksize=100000,
                           usecols=['HADM_ID', 'CATEGORY', 'TEXT'])

cardiac_notes_list = []
for chunk in notes_chunks:
    chunk.columns = chunk.columns.str.lower()
    filtered_notes = chunk[(chunk['hadm_id'].isin(cardiac_admissions_ids)) &
                           (chunk['category'] == 'Discharge summary')]
    cardiac_notes_list.append(filtered_notes)

df_notes = pd.concat(cardiac_notes_list, ignore_index=True)
df_notes = df_notes.drop_duplicates(subset=['hadm_id']).rename(columns={'text': 'reports'})

print("Step 5: Processing procedures...")
proc_with_title = pd.merge(proc, d_proc, on='icd_code', how='left')
cardiac_procs = proc_with_title[proc_with_title['hadm_id'].isin(cardiac_admissions_ids)]

proc_counts = cardiac_procs.groupby('hadm_id').size().reset_index(name='visit_procedure_count')
# Safely join strings together using our newly verified 'long_title'
proc_strings = cardiac_procs.groupby('hadm_id')['long_title'].apply(lambda x: ' | '.join(set(x.dropna()))).reset_index()

print("Step 6: Final Consolidation for Phase 1...")
df_master = pd.merge(df_master, df_notes[['hadm_id', 'reports']], on='hadm_id', how='left')
df_master = pd.merge(df_master, proc_counts, on='hadm_id', how='left')
df_master = pd.merge(df_master, proc_strings, on='hadm_id', how='left')

df_master['visit_procedure_count'] = df_master['visit_procedure_count'].fillna(0).astype(int)
df_master['reports'] = df_master['reports'].fillna('')
df_master['long_title'] = df_master['long_title'].fillna('')

# Save interim step
df_master.to_csv(base_path + 'mimic_iii_heart_clinical_master.csv', index=False)
print(f"Phase 1 complete smoothly! Interim Shape: {df_master.shape}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Step 1: Loading core administrative and mapping files...
Step 2: Isolating the cardiac cohort admissions...
Step 3: Calculating clinical baselines...
Step 4: Extracting discharge summaries from NOTEVENTS (Processing in chunks)...
Step 5: Processing procedures...
Step 6: Final Consolidation for Phase 1...
Phase 1 complete smoothly! Interim Shape: (36396, 33)


In [ ]:
df_master.head()

,row_id_x,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admission_location,discharge_location,insurance,...,dod_hosp,dod_ssn,expire_flag,anchor_age,is_deceased,long_title,icd_code,reports,visit_procedure_count,0
0,22,23,152223,2153-09-03 07:15:00,2153-09-08 19:10:00,NaN,ELECTIVE,PHYS REFERRAL/NORMAL DELI,HOME HEALTH CARE,Medicare,...,NaN,NaN,0,71,0,Coronary atherosclerosis of native coronary ar...,41401,Admission Date: [**2153-9-3**] D...,7,long_title
1,23,23,124321,2157-10-18 19:34:00,2157-10-25 14:00:00,NaN,EMERGENCY,TRANSFER FROM HOSP/EXTRAM,HOME HEALTH CARE,Medicare,...,NaN,NaN,0,75,0,Benign neoplasm of cerebral meninges,2252,Admission Date: [**2157-10-18**] ...,1,long_title
2,24,24,161859,2139-06-06 16:14:00,2139-06-09 12:48:00,NaN,EMERGENCY,TRANSFER FROM HOSP/EXTRAM,HOME,Private,...,NaN,NaN,0,39,0,Acute myocardial infarction of other inferior ...,41041,Admission Date: [**2139-6-6**] D...,6,long_title
3,25,25,129635,2160-11-02 02:06:00,2160-11-05 14:55:00,NaN,EMERGENCY,EMERGENCY ROOM ADMIT,HOME,Private,...,NaN,NaN,0,59,0,"Subendocardial infarction, initial episode of ...",41071,Admission Date: [**2160-11-2**] ...,9,long_title
4,26,26,197661,2126-05-06 15:16:00,2126-05-13 15:00:00,NaN,EMERGENCY,TRANSFER FROM HOSP/EXTRAM,HOME,Medicare,...,NaN,2128-02-25 00:00:00,1,72,0,Mechanical complication of automatic implantab...,99604,Admission Date: [**2126-5-6**] Discharge ...,2,long_title


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

print("Step 6: Executing NLP Feature Engineering on medical records...")
df = pd.read_csv(base_path + 'mimic_iii_heart_clinical_master.csv')

df['reports'] = df['reports'].fillna('')
df['long_title'] = df['long_title'].fillna('')

# NLP Vectorization on text reports
tfidf_reports = TfidfVectorizer(max_features=50, stop_words='english', ngram_range=(1, 2))
reports_matrix = tfidf_reports.fit_transform(df['reports']).toarray()
df_reports_nlp = pd.DataFrame(reports_matrix, columns=[f"report_{w.replace(' ', '_')}" for w in tfidf_reports.get_feature_names_out()])

# NLP Vectorization on procedure titles
tfidf_titles = TfidfVectorizer(max_features=30, stop_words='english', ngram_range=(1, 2))
titles_matrix = tfidf_titles.fit_transform(df['long_title']).toarray()
df_titles_nlp = pd.DataFrame(titles_matrix, columns=[f"title_{w.replace(' ', '_')}" for w in tfidf_titles.get_feature_names_out()])

# Align features via concatenation
df_core = df.drop(columns=['reports', 'long_title']).reset_index(drop=True)
df_nlp_final = pd.concat([df_core, df_reports_nlp, df_titles_nlp], axis=1)

df_nlp_final.to_csv(base_path + 'mimic_iii_heart_nlp_ready.csv', index=False)
print(f"NLP complete! Expanded Matrix Shape: {df_nlp_final.shape}")

Step 6: Executing NLP Feature Engineering on medical records...


NameError: name 'base_path' is not defined

In [ ]:
import pandas as pd
import numpy as np
import os

base_path = '/content/drive/MyDrive/MIMIC_III_REAL/'

print("Step 7: Beginning large-scale chunked parsing of CHARTEVENTS vital signs...")
df_master = pd.read_csv(base_path + 'mimic_iii_heart_nlp_ready.csv')
target_hadms = set(df_master['hadm_id'].unique())

# Comprehensive MIMIC-III ItemID dictionary mapping (CareVue & Metavision)
vitals_mapping = {
    # Heart Rate
    211: 'vital_mean_heart_rate', 220045: 'vital_mean_heart_rate',
    # Systolic BP
    51: 'vital_mean_systolic_bp', 442: 'vital_mean_systolic_bp',
    455: 'vital_mean_systolic_bp', 220179: 'vital_mean_systolic_bp',
    # Diastolic BP
    8368: 'vital_mean_diastolic_bp', 8440: 'vital_mean_diastolic_bp',
    8441: 'vital_mean_diastolic_bp', 220180: 'vital_mean_diastolic_bp',
    # SpO2
    646: 'vital_mean_spo2', 220277: 'vital_mean_spo2',
    # Temp
    678: 'vital_mean_temp', 223761: 'vital_mean_temp'
}

# Explicitly set the file path using the exact name from your Drive screenshot
target_file = base_path + 'CHARTEVENTS.csv (1).gz'

# Fallback auto-checks just in case the name shifts
if not os.path.exists(target_file):
    if os.path.exists(base_path + 'CHARTEVENTS.csv.gz'):
        target_file = base_path + 'CHARTEVENTS.csv.gz'
    elif os.path.exists(base_path + 'chartevents.csv.gz'):
        target_file = base_path + 'chartevents.csv.gz'

print(f"Verified target file path being read: {target_file}")

chunk_container = []
# Process chartevents in 500k row chunks to maximize RAM safety boundaries
chartevents_stream = pd.read_csv(target_file, compression='gzip', chunksize=500000,
                                 usecols=['HADM_ID', 'ITEMID', 'VALUENUM'])

for i, chunk in enumerate(chartevents_stream):
    chunk.columns = chunk.columns.str.lower()

    # Filter for our specific cohort and target vitals
    filtered_chunk = chunk[(chunk['hadm_id'].isin(target_hadms)) & (chunk['itemid'].isin(vitals_mapping.keys()))]
    if not filtered_chunk.empty:
        # Map itemid integers to clear column name strings
        filtered_chunk['vital_name'] = filtered_chunk['itemid'].map(vitals_mapping)
        chunk_container.append(filtered_chunk[['hadm_id', 'vital_name', 'valuenum']])

    if i % 20 == 0:
        print(f"Processed {i * 500000} rows through streaming parser...")

print("Pivoting data frames into summary vital metrics matrices...")
df_vitals_raw = pd.concat(chunk_container, ignore_index=True)

# Group, pivot, and average out values per admission
df_vitals_pivoted = df_vitals_raw.groupby(['hadm_id', 'vital_name'])['valuenum'].mean().unstack().reset_index()

# Merge back with left join to retain all rows from your NLP master
final_dataset = pd.merge(df_master, df_vitals_pivoted, on='hadm_id', how='left')

# Aligning exactly with your clean MIMIC-IV feature layout
keep_cols = [
    'subject_id', 'hadm_id', 'icd_code', 'gender', 'anchor_age',
    'hospital_expire_flag', 'is_deceased', 'visit_procedure_count',
    'vital_mean_heart_rate', 'vital_mean_systolic_bp', 'vital_mean_diastolic_bp',
    'vital_mean_spo2', 'vital_mean_temp'
]
# Gather the dynamically generated text features from your previous NLP step
nlp_cols = [col for col in final_dataset.columns if col.startswith('report_') or col.startswith('title_')]
final_columns_to_export = keep_cols + nlp_cols

df_output = final_dataset[[c for c in final_columns_to_export if c in final_dataset.columns]].copy()

# Save final structured matrix to your drive
output_filename = base_path + 'final_mimic_iii_heart_clinical_master.csv'
df_output.to_csv(output_filename, index=False)

print("\n🎉 SUCCESS! DATA GENERATION COMPLETE.")
print(f"Final Data Matrix Shape: {df_output.shape}")
print(f"Saved complete dataset as: {output_filename}")

Step 7: Beginning large-scale chunked parsing of CHARTEVENTS vital signs...
Verified target file path being read: /content/drive/MyDrive/MIMIC_III_REAL/CHARTEVENTS.csv (1).gz


/tmp/ipykernel_1908/1173364449.py:51: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_chunk['vital_name'] = filtered_chunk['itemid'].map(vitals_mapping)


Processed 0 rows through streaming parser...


/tmp/ipykernel_1908/1173364449.py:51: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_chunk['vital_name'] = filtered_chunk['itemid'].map(vitals_mapping)
/tmp/ipykernel_1908/1173364449.py:51: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_chunk['vital_name'] = filtered_chunk['itemid'].map(vitals_mapping)
/tmp/ipykernel_1908/1173364449.py:51: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See t

Processed 10000000 rows through streaming parser...


...

Processed 240000000 rows through streaming parser...


/tmp/ipykernel_1908/1173364449.py:51: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_chunk['vital_name'] = filtered_chunk['itemid'].map(vitals_mapping)
/tmp/ipykernel_1908/1173364449.py:51: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_chunk['vital_name'] = filtered_chunk['itemid'].map(vitals_mapping)
/tmp/ipykernel_1908/1173364449.py:51: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See t

Processed 250000000 rows through streaming parser...


/tmp/ipykernel_1908/1173364449.py:51: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_chunk['vital_name'] = filtered_chunk['itemid'].map(vitals_mapping)
/tmp/ipykernel_1908/1173364449.py:51: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_chunk['vital_name'] = filtered_chunk['itemid'].map(vitals_mapping)
/tmp/ipykernel_1908/1173364449.py:51: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See t

Processed 260000000 rows through streaming parser...
Processed 270000000 rows through streaming parser...
Processed 280000000 rows through streaming parser...
Processed 290000000 rows through streaming parser...
Processed 300000000 rows through streaming parser...
Processed 310000000 rows through streaming parser...
Processed 320000000 rows through streaming parser...
Processed 330000000 rows through streaming parser...
Pivoting data frames into summary vital metrics matrices...

🎉 SUCCESS! DATA GENERATION COMPLETE.
Final Data Matrix Shape: (36396, 93)
Saved complete dataset as: /content/drive/MyDrive/MIMIC_III_REAL/final_mimic_iii_heart_clinical_master.csv


In [ ]:
import pandas as pd
import numpy as np

# 1. Setup active path targeting your current MIMIC_III_REAL drive folder
base_path = '/content/drive/MyDrive/MIMIC_III_REAL/'

print("Loading the model-ready dataset...")
# Load the matrix generated right before the vitals step
df = pd.read_csv(base_path + 'mimic_iii_heart_nlp_ready.csv')

# 2. Clean the icd column to ensure consistent matching
df['icd_clean'] = df['icd_code'].astype(str).str.strip()

# 3. Comprehensive clinical mapping dictionary for your dataset's codes
icd_text_mapping = {
    # --- Major ICD-9 Cardiovascular & Renal Codes ---
    '40391': 'Hypertensive chronic kidney disease, stage 5 or end-stage renal disease',
    '4019': 'Unspecified essential hypertension',
    '42833': 'Acute on chronic diastolic heart failure',
    '4280': 'Congestive heart failure, unspecified',
    '41401': 'Coronary atherosclerosis of native coronary artery',
    '41071': 'Subendocardial infarction, initial episode of care',
    '412': 'Old myocardial infarction',
    '42731': 'Atrial fibrillation',
    '4241': 'Aortic valve disorders',
    '4240': 'Mitral valve disorders',

    # --- Major ICD-10 Cardiovascular Codes ("I-Codes") ---
    'I10': 'Essential (primary) hypertension',
    'I110': 'Hypertensive heart disease with heart failure',
    'I120': 'Hypertensive chronic kidney disease with stage 5 or end-stage renal disease',
    'I214': 'Acute subendocardial myocardial infarction',
    'I2510': 'Atherosclerotic heart disease of native coronary artery without angina pectoris',
    'I480': 'Paroxysmal atrial fibrillation',
    'I481': 'Persistent atrial fibrillation',
    'I482': 'Chronic atrial fibrillation',
    'I509': 'Heart failure, unspecified',
    'I5033': 'Acute on chronic diastolic (congestive) heart failure',
    'I5023': 'Acute on chronic systolic (congestive) heart failure'
}

# 4. Apply the mapping to create a new text column
def get_icd_description(code):
    if code in icd_text_mapping:
        return icd_text_mapping[code]
    elif str(code).startswith('I'):
        return f"ICD-10 Circulatory/Cardiac Condition (Code: {code})"
    else:
        return f"ICD-9 Cardiovascular Condition (Code: {code})"

print("Applying medical text description mapping layer...")
df['icd_description'] = df['icd_clean'].apply(get_icd_description)

# 5. Clean up and structural reordering so description sits next to the code identifier
df = df.drop(columns=['icd_clean'])
cols = list(df.columns)

# Safely extract and move 'icd_description' right next to 'icd_code'
if 'icd_code' in cols and 'icd_description' in cols:
    icd_idx = cols.index('icd_code')
    cols.insert(icd_idx + 1, cols.pop(cols.index('icd_description')))
    df = df[cols]

# Save the newly transformed dataset back to your drive files directory
output_file = base_path + 'final_heart_clinical_master_with_text.csv'
df.to_csv(output_file, index=False)

print("\n🎉 Transformation Complete! Sample of your new text descriptions:")
print(df[['subject_id', 'hadm_id', 'icd_code', 'icd_description']].head(10))

Loading the model-ready dataset...
Applying medical text description mapping layer...

🎉 Transformation Complete! Sample of your new text descriptions:
   subject_id  hadm_id icd_code  \
0          23   152223    41401   
1          23   124321     2252   
2          24   161859    41041   
3          25   129635    41071   
4          26   197661    99604   
5          28   162569    41401   
6          30   104557    99674   
7          34   115799    41071   
8          34   144319    42789   
9          35   166707     3962   

                                     icd_description  
0  Coronary atherosclerosis of native coronary ar...  
1        ICD-9 Cardiovascular Condition (Code: 2252)  
2       ICD-9 Cardiovascular Condition (Code: 41041)  
3  Subendocardial infarction, initial episode of ...  
4       ICD-9 Cardiovascular Condition (Code: 99604)  
5  Coronary atherosclerosis of native coronary ar...  
6       ICD-9 Cardiovascular Condition (Code: 99674)  
7  Subendocardial infa

In [ ]:
df.head()

,row_id_x,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admission_location,discharge_location,insurance,...,title_malignant,title_mention,title_native,title_native_coronary,title_septicemia,title_subendocardial,title_subendocardial_infarction,title_unspecified,title_unspecified_septicemia,title_valve
0,22,23,152223,2153-09-03 07:15:00,2153-09-08 19:10:00,NaN,ELECTIVE,PHYS REFERRAL/NORMAL DELI,HOME HEALTH CARE,Medicare,...,0.0,0.0,0.298999,0.306371,0.0,0.000000,0.000000,0.0,0.0,0.0
1,23,23,124321,2157-10-18 19:34:00,2157-10-25 14:00:00,NaN,EMERGENCY,TRANSFER FROM HOSP/EXTRAM,HOME HEALTH CARE,Medicare,...,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0
2,24,24,161859,2139-06-06 16:14:00,2139-06-09 12:48:00,NaN,EMERGENCY,TRANSFER FROM HOSP/EXTRAM,HOME,Private,...,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0
3,25,25,129635,2160-11-02 02:06:00,2160-11-05 14:55:00,NaN,EMERGENCY,EMERGENCY ROOM ADMIT,HOME,Private,...,0.0,0.0,0.000000,0.000000,0.0,0.374818,0.374818,0.0,0.0,0.0
4,26,26,197661,2126-05-06 15:16:00,2126-05-13 15:00:00,NaN,EMERGENCY,TRANSFER FROM HOSP/EXTRAM,HOME,Medicare,...,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0


In [ ]:
df.isnull().sum()

,0
row_id_x,0
subject_id,0
hadm_id,0
admittime,0
dischtime,0
...,...
title_subendocardial,0
title_subendocardial_infarction,0
title_unspecified,0
title_unspecified_septicemia,0


In [ ]:
import pandas as pd
import numpy as np

# 1. Setup path targeting your current MIMIC_III_REAL drive folder
base_path = '/content/drive/MyDrive/MIMIC_III_REAL/'

print("Loading final mapped heart master dataset...")
df = pd.read_csv(base_path + 'final_heart_clinical_master_with_text.csv')

# 2. CATEGORICAL ENCODING
print("Encoding categorical tracking features...")

# Create 'is_male': 'M' becomes 1, 'F' becomes 0
if 'gender' in df.columns:
    df['is_male'] = df['gender'].map({'M': 1, 'F': 0})
    df['is_male'] = df['is_male'].fillna(df['is_male'].mode()[0]).astype(int)

# 3. SELECTIVE FEATURE REMOVAL (Keeping 'icd_description' exactly as requested)
# We are only removing pure administrative noise and duplicate flags
features_to_drop = [
    'row_id_x', 'row_id_y',         # Row counters from database merges
    # 'subject_id', 'hadm_id',        # Internal database tracking integers
    'gender',                       # Removed because it is replaced by 'is_male'
    # 'hospital_expire_flag',         # Handled: this is your raw target variable
    'is_deceased', 'expire_flag',   # Redundant duplicates of your target flag
    'admittime', 'dischtime',       # Raw un-parsed date strings
    'deathtime', 'dod', 'dod_hosp', # Leaky timestamp metrics containing post-event answers
    'dod_ssn', 'stay_window',       # Administrative date records
    'admission_type', 'insurance',  # Administrative demographics
    'admission_location',
    'discharge_location',
    'language', 'religion',
    'marital_status', 'ethnicity',
    'edregtime', 'edouttime',
    'has_chartevents_data'
]

# Drop administrative entries safely
df_ml_ready = df.drop(columns=features_to_drop, errors='ignore')

# 4. HANDLE MISSING VALUES IN CLINICAL METRICS
print("Handling null values in quantitative clinical features...")
# Make sure we don't accidentally impute text strings or our main classification label
skip_columns = ['hospital_expire_flag', 'icd_description', 'icd_code', 'diagnosis']

for col in df_ml_ready.columns:
    if col not in skip_columns and df_ml_ready[col].dtype in [np.float64, np.int64]:
        median_val = df_ml_ready[col].median()
        # Fallback to zero if the column is completely empty
        if pd.isna(median_val):
            median_val = 0
        df_ml_ready[col] = df_ml_ready[col].fillna(median_val)

# 5. RE-ARRANGE TARGET TO THE END FOR EASY ACCESS
# Ensure your target flag is the final column in the sheet layout
if 'hospital_expire_flag' in df.columns:
    df_ml_ready['hospital_expire_flag'] = df['hospital_expire_flag'].fillna(0).astype(int)

# 6. SAVE COMPLETED MATRIX
output_file = base_path + 'MIMIC_III_CARDIAC_ML_READY.csv'
df_ml_ready.to_csv(output_file, index=False)

print("\n🎉 Preprocessing Complete!")
print(f"Final Data Matrix Shape for ML: {df_ml_ready.shape}")
print(f"Target Label ('hospital_expire_flag') Balance:\n{df_ml_ready['hospital_expire_flag'].value_counts()}")
print("\nVerified that 'icd_description' is preserved in your dataset columns.")

Loading final mapped heart master dataset...
Encoding categorical tracking features...
Handling null values in quantitative clinical features...

🎉 Preprocessing Complete!
Final Data Matrix Shape for ML: (36396, 91)
Target Label ('hospital_expire_flag') Balance:
hospital_expire_flag
0    31451
1     4945
Name: count, dtype: int64

Verified that 'icd_description' is preserved in your dataset columns.


In [10]:
import pandas as pd
import numpy as np

base_path = '/content/drive/MyDrive/MIMIC_III_REAL/'

print("Step 1: Loading your calculated vital sign baseline matrix...")
# Load the dataset generated in Step 7 that actually contains your 5 vitals columns
df_vitals = pd.read_csv(base_path + 'final_mimic_iii_heart_clinical_master.csv')

# 2. Clean the icd column to ensure consistent matching
df_vitals['icd_clean'] = df_vitals['icd_code'].astype(str).str.strip()

# 3. Your clinical mapping dictionary matching the MIMIC-IV layout
icd_text_mapping = {
    '40391': 'Hypertensive chronic kidney disease, stage 5 or end-stage renal disease',
    '4019': 'Unspecified essential hypertension',
    '42833': 'Acute on chronic diastolic heart failure',
    '4280': 'Congestive heart failure, unspecified',
    '41401': 'Coronary atherosclerosis of native coronary artery',
    '41071': 'Subendocardial infarction, initial episode of care',
    '412': 'Old myocardial infarction',
    '42731': 'Atrial fibrillation',
    '4241': 'Aortic valve disorders',
    '4240': 'Mitral valve disorders',
    'I10': 'Essential (primary) hypertension',
    'I110': 'Hypertensive heart disease with heart failure',
    'I120': 'Hypertensive chronic kidney disease with stage 5 or end-stage renal disease',
    'I214': 'Acute subendocardial myocardial infarction',
    'I2510': 'Atherosclerotic heart disease of native coronary artery without angina pectoris',
    'I480': 'Paroxysmal atrial fibrillation',
    'I481': 'Persistent atrial fibrillation',
    'I482': 'Chronic atrial fibrillation',
    'I509': 'Heart failure, unspecified',
    'I5033': 'Acute on chronic diastolic (congestive) heart failure',
    'I5023': 'Acute on chronic systolic (congestive) heart failure'
}

def get_icd_description(code):
    if code in icd_text_mapping:
        return icd_text_mapping[code]
    elif str(code).startswith('I'):
        return f"ICD-10 Circulatory/Cardiac Condition (Code: {code})"
    else:
        return f"ICD-9 Cardiovascular Condition (Code: {code})"

print("Step 2: Merging the 'icd_description' text feature column...")
df_vitals['icd_description'] = df_vitals['icd_clean'].apply(get_icd_description)
df_vitals = df_vitals.drop(columns=['icd_clean'])

# Reorder columns so description sits nicely next to the code identifier
cols = list(df_vitals.columns)
if 'icd_code' in cols and 'icd_description' in cols:
    icd_idx = cols.index('icd_code')
    cols.insert(icd_idx + 1, cols.pop(cols.index('icd_description')))
    df_vitals = df_vitals[cols]

# 4. ENCODE GENDER Flag to 'is_male'
print("Step 3: Appending engineered demographic numeric flags...")
if 'gender' in df_vitals.columns:
    df_vitals['is_male'] = df_vitals['gender'].map({'M': 1, 'F': 0})
    df_vitals['is_male'] = df_vitals['is_male'].fillna(df_vitals['is_male'].mode()[0]).astype(int)

# 5. RE-ARRANGE TARGET TO THE END
if 'hospital_expire_flag' in df_vitals.columns:
    df_vitals['hospital_expire_flag'] = df_vitals['hospital_expire_flag'].fillna(0).astype(int)

# Overwrite the final ML file so your training scripts load everything perfectly
output_ml_file = base_path + 'MIMIC_III_CARDIAC_ML_READY.csv'
df_vitals.to_csv(output_ml_file, index=False)

print("\n🎉 PATCH SUCCESSFUL!")
print(f"Verified Final Matrix Shape: {df_vitals.shape}")
print("Columns successfully integrated and aligned for Modeling.")

Step 1: Loading your calculated vital sign baseline matrix...
Step 2: Merging the 'icd_description' text feature column...
Step 3: Appending engineered demographic numeric flags...

🎉 PATCH SUCCESSFUL!
Verified Final Matrix Shape: (36396, 95)
Columns successfully integrated and aligned for Modeling.


In [3]:
import pandas as pd
import numpy as np
import re
from google.colab import drive
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

# 1. Mount Drive and Setup Paths to your active MIMIC-III directory
drive.mount('/content/drive', force_remount=True)
base_path = '/content/drive/MyDrive/MIMIC_III_REAL/'

print("Loading your processed MIMIC-III training matrix...")
# Load the ML-ready file we just generated with all your text features and icd_description intact
df = pd.read_csv(base_path + 'MIMIC_III_CARDIAC_ML_READY.csv')

# 2. Separate Target and Group Identifiers
# Setting your explicit target to hospital_expire_flag
y = df['hospital_expire_flag'].astype(int)
groups = df['subject_id']

# 3. Handle Features and One-Hot Encoding
# Exclude structural identifiers and the raw target column from feature space X
cols_to_exclude = ['subject_id', 'hadm_id', 'stay_id', 'icd_code', 'hospital_expire_flag', 'diagnosis']
X_raw = df.drop(columns=[col for col in cols_to_exclude if col in df.columns])

# One-hot encode the icd_description column that you wanted to preserve
print("One-hot encoding medical descriptions...")
X_encoded = pd.get_dummies(X_raw, columns=['icd_description'], prefix='Dx', prefix_sep='_')

# Clean column headers so special characters from descriptions don't crash ML algorithms
X_encoded = X_encoded.rename(columns=lambda x: re.sub(r'[^A-Za-z0-9_]+', '', x))

# 4. Define Continuous Clinical Features for Scaling
continuous_cols = [
    'anchor_age', 'visit_procedure_count', 'vital_mean_heart_rate',
    'vital_mean_systolic_bp', 'vital_mean_diastolic_bp',
    'vital_mean_spo2', 'vital_mean_temp'
]

# Safeguard check: only scale columns if they made it into the encoded feature matrix
continuous_cols = [col for col in continuous_cols if col in X_encoded.columns]

# 5. GroupKFold Splitting (5-Fold Cross-Validation Setup)
print("Splitting datasets using GroupKFold via patient subject IDs...")
gkf = GroupKFold(n_splits=5)
train_idx, test_idx = next(gkf.split(X_encoded, y, groups=groups))

X_train, X_test = X_encoded.iloc[train_idx].copy(), X_encoded.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# 6. Scale Continuous Features
print("Normalizing continuous clinical metrics...")
scaler = StandardScaler()
X_train[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])
X_test[continuous_cols] = scaler.transform(X_test[continuous_cols])

print("\n🎉 PIPELINE EXECUTION SUCCESSFUL!")
print(f"X_train Shape: {X_train.shape} | X_test Shape: {X_test.shape}")
print(f"Total Unique Patient Groups Kept Intact: {groups.nunique()}")

Mounted at /content/drive
Loading your processed MIMIC-III training matrix...
One-hot encoding medical descriptions...
Splitting datasets using GroupKFold via patient subject IDs...
Normalizing continuous clinical metrics...

🎉 PIPELINE EXECUTION SUCCESSFUL!
X_train Shape: (29116, 2114) | X_test Shape: (7280, 2114)
Total Unique Patient Groups Kept Intact: 28222


In [4]:
import pandas as pd
import numpy as np
import re
from google.colab import drive
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

# 1. Mount Drive and Setup Paths to your active MIMIC-III directory
drive.mount('/content/drive', force_remount=True)
base_path = '/content/drive/MyDrive/MIMIC_III_REAL/'

print("Loading your processed MIMIC-III training matrix...")
df = pd.read_csv(base_path + 'MIMIC_III_CARDIAC_ML_READY.csv')

# 2. Separate Target and Group Identifiers
y = df['hospital_expire_flag'].astype(int)
groups = df['subject_id']

# 3. Handle Features and One-Hot Encoding
# ADDED EXTRA PROTECTION: Explicitly exclude all date, timestamp, and text tracker names
cols_to_exclude = [
    'subject_id', 'hadm_id', 'stay_id', 'icd_code', 'hospital_expire_flag',
    'diagnosis', 'admittime', 'dischtime', 'deathtime', 'dob', 'dod',
    'dod_hosp', 'dod_ssn', 'stay_window', 'edregtime', 'edouttime'
]
X_raw = df.drop(columns=[col for col in cols_to_exclude if col in df.columns])

# One-hot encode the icd_description column that you wanted to preserve
print("One-hot encoding medical descriptions...")
X_encoded = pd.get_dummies(X_raw, columns=['icd_description'], prefix='Dx', prefix_sep='_')

# Clean column headers so special characters from descriptions don't crash ML algorithms
X_encoded = X_encoded.rename(columns=lambda x: re.sub(r'[^A-Za-z0-9_]+', '', x))

# Double check that no object/string columns remain in our math matrix
non_numeric_cols = X_encoded.select_dtypes(exclude=[np.number]).columns
if len(non_numeric_cols) > 0:
    print(f"⚠️ Dropping remaining text-string columns to prevent fits crash: {list(non_numeric_cols)}")
    X_encoded = X_encoded.drop(columns=non_numeric_cols)

# 4. Define Continuous Clinical Features for Scaling
continuous_cols = [
    'anchor_age', 'visit_procedure_count', 'vital_mean_heart_rate',
    'vital_mean_systolic_bp', 'vital_mean_diastolic_bp',
    'vital_mean_spo2', 'vital_mean_temp'
]

# Safeguard check: only scale columns if they made it into the encoded feature matrix
continuous_cols = [col for col in continuous_cols if col in X_encoded.columns]

# 5. GroupKFold Splitting (5-Fold Cross-Validation Setup)
print("Splitting datasets using GroupKFold via patient subject IDs...")
gkf = GroupKFold(n_splits=5)
train_idx, test_idx = next(gkf.split(X_encoded, y, groups=groups))

X_train, X_test = X_encoded.iloc[train_idx].copy(), X_encoded.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# Save tracking variables for the modeling cell
groups_train = groups.iloc[train_idx]

# 6. Scale Continuous Features
print("Normalizing continuous clinical metrics...")
scaler = StandardScaler()
X_train[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])
X_test[continuous_cols] = scaler.transform(X_test[continuous_cols])

print("\n🎉 PIPELINE SETUP SUCCESSFUL!")
print(f"X_train Shape: {X_train.shape} | X_test Shape: {X_test.shape}")

Mounted at /content/drive
Loading your processed MIMIC-III training matrix...
One-hot encoding medical descriptions...
⚠️ Dropping remaining text-string columns to prevent fits crash: ['gender', 'Dx_Acuteonchronicdiastolicheartfailure', 'Dx_Aorticvalvedisorders', 'Dx_Atrialfibrillation', 'Dx_Congestiveheartfailureunspecified', 'Dx_Coronaryatherosclerosisofnativecoronaryartery', 'Dx_Hypertensivechronickidneydiseasestage5orendstagerenaldisease', 'Dx_ICD9CardiovascularConditionCode0031', 'Dx_ICD9CardiovascularConditionCode0041', 'Dx_ICD9CardiovascularConditionCode0051', 'Dx_ICD9CardiovascularConditionCode0059', 'Dx_ICD9CardiovascularConditionCode00845', 'Dx_ICD9CardiovascularConditionCode0085', 'Dx_ICD9CardiovascularConditionCode00863', 'Dx_ICD9CardiovascularConditionCode00869', 'Dx_ICD9CardiovascularConditionCode0088', 'Dx_ICD9CardiovascularConditionCode0090', 'Dx_ICD9CardiovascularConditionCode0091', 'Dx_ICD9CardiovascularConditionCode0093', 'Dx_ICD9CardiovascularConditionCode01205', 'D

In [6]:
import pandas as pd
import numpy as np
import re
from google.colab import drive
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

# 1. Mount Drive and Setup Paths to your active MIMIC-III directory
drive.mount('/content/drive', force_remount=True)
base_path = '/content/drive/MyDrive/MIMIC_III_REAL/'

print("Loading your processed MIMIC-III training matrix...")
df = pd.read_csv(base_path + 'MIMIC_III_CARDIAC_ML_READY.csv')

# 2. Separate Target and Group Identifiers
y = df['hospital_expire_flag'].astype(int)
groups = df['subject_id']

# 3. Handle Features and One-Hot Encoding
cols_to_exclude = [
    'subject_id', 'hadm_id', 'stay_id', 'icd_code', 'hospital_expire_flag',
    'diagnosis', 'admittime', 'dischtime', 'deathtime', 'dob', 'dod',
    'dod_hosp', 'dod_ssn', 'stay_window', 'edregtime', 'edouttime'
]
X_raw = df.drop(columns=[col for col in cols_to_exclude if col in df.columns])

print("One-hot encoding medical descriptions...")
X_encoded = pd.get_dummies(X_raw, columns=['icd_description'], prefix='Dx', prefix_sep='_')

# Clean column headers
X_encoded = X_encoded.rename(columns=lambda x: re.sub(r'[^A-Za-z0-9_]+', '', x))

# ==========================================
# 🔥 FIXED LAYER: IMPUTE RECENTLY LEAKED NANs
# ==========================================
print("Imputing any missing values found in features matrix...")
# For any patient admission missing vitals, fill with the median value of that specific vital sign
for col in X_encoded.columns:
    if X_encoded[col].isnull().any():
        median_val = X_encoded[col].median()
        # Fallback handle if a column is entirely null
        if pd.isna(median_val):
            median_val = 0
        X_encoded[col] = X_encoded[col].fillna(median_val)
# ==========================================

# 4. Define Continuous Clinical Features for Scaling
continuous_cols = [
    'anchor_age', 'visit_procedure_count', 'vital_mean_heart_rate',
    'vital_mean_systolic_bp', 'vital_mean_diastolic_bp',
    'vital_mean_spo2', 'vital_mean_temp'
]
continuous_cols = [col for col in continuous_cols if col in X_encoded.columns]

# 5. GroupKFold Splitting
print("Splitting datasets using GroupKFold via patient subject IDs...")
gkf = GroupKFold(n_splits=5)
train_idx, test_idx = next(gkf.split(X_encoded, y, groups=groups))

X_train, X_test = X_encoded.iloc[train_idx].copy(), X_encoded.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# Save tracking variables for the modeling cell
groups_train = groups.iloc[train_idx]

# 6. Scale Continuous Features
print("Normalizing continuous clinical metrics...")
scaler = StandardScaler()
X_train[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])
X_test[continuous_cols] = scaler.transform(X_test[continuous_cols])

print("\n🎉 PIPELINE SETUP SUCCESSFUL! No NaN values remaining.")
print(f"X_train Shape: {X_train.shape} | X_test Shape: {X_test.shape}")

Mounted at /content/drive
Loading your processed MIMIC-III training matrix...
One-hot encoding medical descriptions...
Imputing any missing values found in features matrix...
Splitting datasets using GroupKFold via patient subject IDs...
Normalizing continuous clinical metrics...

🎉 PIPELINE SETUP SUCCESSFUL! No NaN values remaining.
X_train Shape: (29116, 2114) | X_test Shape: (7280, 2114)


In [8]:
import pandas as pd
import numpy as np
import re
from google.colab import drive
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

# 1. Mount Drive and Setup Paths to your active MIMIC-III directory
drive.mount('/content/drive', force_remount=True)
base_path = '/content/drive/MyDrive/MIMIC_III_REAL/'

print("Loading your processed MIMIC-III training matrix...")
df = pd.read_csv(base_path + 'MIMIC_III_CARDIAC_ML_READY.csv')

# 2. Separate Target and Group Identifiers
y = df['hospital_expire_flag'].astype(int)
groups = df['subject_id']

# 3. Handle Features and One-Hot Encoding
# CRITICAL FIX: Explicitly added 'gender' to the exclude list so 'M' and 'F' strings are eliminated!
cols_to_exclude = [
    'subject_id', 'hadm_id', 'stay_id', 'icd_code', 'hospital_expire_flag',
    'diagnosis', 'gender', 'admittime', 'dischtime', 'deathtime', 'dob', 'dod',
    'dod_hosp', 'dod_ssn', 'stay_window', 'edregtime', 'edouttime'
]
X_raw = df.drop(columns=[col for col in cols_to_exclude if col in df.columns])

print("One-hot encoding medical descriptions...")
# This safely handles 'icd_description' into binary numeric flags
X_encoded = pd.get_dummies(X_raw, columns=['icd_description'], prefix='Dx', prefix_sep='_')

# Clean column headers
X_encoded = X_encoded.rename(columns=lambda x: re.sub(r'[^A-Za-z0-9_]+', '', x))

# Double check and force-drop any stray non-numeric text columns just in case
stray_text_cols = X_encoded.select_dtypes(exclude=[np.number]).columns
if len(stray_text_cols) > 0:
    print(f"⚠️ Dropping remaining text-string columns to protect training: {list(stray_text_cols)}")
    X_encoded = X_encoded.drop(columns=stray_text_cols)

print("Imputing any missing values found in features matrix...")
# For any patient admission missing values, fill with the column median
for col in X_encoded.columns:
    if X_encoded[col].isnull().any():
        median_val = X_encoded[col].median()
        if pd.isna(median_val):
            median_val = 0
        X_encoded[col] = X_encoded[col].fillna(median_val)

# 4. Define Continuous Clinical Features for Scaling
continuous_cols = [
    'anchor_age', 'visit_procedure_count', 'vital_mean_heart_rate',
    'vital_mean_systolic_bp', 'vital_mean_diastolic_bp',
    'vital_mean_spo2', 'vital_mean_temp'
]
continuous_cols = [col for col in continuous_cols if col in X_encoded.columns]

# 5. GroupKFold Splitting
print("Splitting datasets using GroupKFold via patient subject IDs...")
gkf = GroupKFold(n_splits=5)
train_idx, test_idx = next(gkf.split(X_encoded, y, groups=groups))

X_train, X_test = X_encoded.iloc[train_idx].copy(), X_encoded.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# Save tracking variables for the modeling cell
groups_train = groups.iloc[train_idx]

# 6. Scale Continuous Features
print("Normalizing continuous clinical metrics...")
scaler = StandardScaler()
X_train[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])
X_test[continuous_cols] = scaler.transform(X_test[continuous_cols])

print("\n🎉 PIPELINE SETUP SUCCESSFUL! No text or NaNs remain inside features.")
print(f"X_train Shape: {X_train.shape} | X_test Shape: {X_test.shape}")

Mounted at /content/drive
Loading your processed MIMIC-III training matrix...
One-hot encoding medical descriptions...
⚠️ Dropping remaining text-string columns to protect training: ['Dx_Acuteonchronicdiastolicheartfailure', 'Dx_Aorticvalvedisorders', 'Dx_Atrialfibrillation', 'Dx_Congestiveheartfailureunspecified', 'Dx_Coronaryatherosclerosisofnativecoronaryartery', 'Dx_Hypertensivechronickidneydiseasestage5orendstagerenaldisease', 'Dx_ICD9CardiovascularConditionCode0031', 'Dx_ICD9CardiovascularConditionCode0041', 'Dx_ICD9CardiovascularConditionCode0051', 'Dx_ICD9CardiovascularConditionCode0059', 'Dx_ICD9CardiovascularConditionCode00845', 'Dx_ICD9CardiovascularConditionCode0085', 'Dx_ICD9CardiovascularConditionCode00863', 'Dx_ICD9CardiovascularConditionCode00869', 'Dx_ICD9CardiovascularConditionCode0088', 'Dx_ICD9CardiovascularConditionCode0090', 'Dx_ICD9CardiovascularConditionCode0091', 'Dx_ICD9CardiovascularConditionCode0093', 'Dx_ICD9CardiovascularConditionCode01205', 'Dx_ICD9Cardio

In [9]:
import pandas as pd
import numpy as np
import re
import warnings
from sklearn.model_selection import GroupKFold, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

# 1. Hyperparameter Search Space Setup
param_distributions = {
    'C': np.logspace(-2, 1, 10),
    'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
}

cv_strategy = GroupKFold(n_splits=5)

# Increased max_iter from 200 to 500 to address the SAGA ConvergenceWarning
base_lr = LogisticRegression(
    penalty='elasticnet',
    solver='saga',
    class_weight='balanced',
    max_iter=500,
    random_state=42
)

print("Running fast RandomizedSearchCV over ElasticNet parameters using MIMIC-III cohort splits...")
lr_search = RandomizedSearchCV(
    estimator=base_lr,
    param_distributions=param_distributions,
    n_iter=8,
    scoring='roc_auc',
    cv=cv_strategy.split(X_train, y_train, groups=groups_train),
    random_state=42,
    n_jobs=-1
)

lr_search.fit(X_train, y_train)

print("\n================ OPTIMIZATION SEARCH RESULTS COMPLETE ================")
print(f"Best Parameters Found -> C: {lr_search.best_params_['C']:.4f} | l1_ratio: {lr_search.best_params_['l1_ratio']:.2f}")

# 2. Evaluate Tuned Estimator on Holdout Test Split
best_model = lr_search.best_estimator_
y_prob_tuned = best_model.predict_proba(X_test)[:, 1]

# 3. Build Evaluation Summary Table Linked by Patient Keys
test_summary = pd.DataFrame({
    'subject_id': df.iloc[test_idx]['subject_id'],
    'y_true': y_test,
    'y_prob': y_prob_tuned
})

# 4. FIX: Collapse down to Patient-Level Summary Metrics safely
# We take the max of y_true so it stays binary (1 if they passed away in any admission, else 0)
patient_res_true = test_summary.groupby('subject_id')['y_true'].max()
patient_res_pred = test_summary.groupby('subject_id')['y_prob'].mean()

# Binary classification cutoff threshold
preds = (patient_res_pred >= 0.5).astype(int)

print("\n--- Optimized MIMIC-III Patient-Level Evaluation ---")
print(classification_report(patient_res_true, preds))
print(f"Optimized Patient-Level ROC-AUC Score: {roc_auc_score(patient_res_true, patient_res_pred):.4f}")

Running fast RandomizedSearchCV over ElasticNet parameters using MIMIC-III cohort splits...

================ OPTIMIZATION SEARCH RESULTS COMPLETE ================
Best Parameters Found -> C: 0.0464 | l1_ratio: 0.70

--- Optimized MIMIC-III Patient-Level Evaluation ---
              precision    recall  f1-score   support

           0       0.97      1.00      0.99      4674
           1       1.00      0.87      0.93       970

    accuracy                           0.98      5644
   macro avg       0.99      0.94      0.96      5644
weighted avg       0.98      0.98      0.98      5644

Optimized Patient-Level ROC-AUC Score: 1.0000


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [13]:
import pandas as pd
import numpy as np
import re
from google.colab import drive
from sklearn.model_selection import GroupKFold, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score

# 1. Load the core dataset
drive.mount('/content/drive', force_remount=True)
base_path = '/content/drive/MyDrive/MIMIC_III_REAL/'

print("Loading dataset...")
df = pd.read_csv(base_path + 'MIMIC_III_CARDIAC_ML_READY.csv')

# 2. Extract targets and patient groups
y = df['hospital_expire_flag'].astype(int)
groups = df['subject_id']

# 3. Drop all raw tracking text, dates, and non-numeric administrative noise
cols_to_exclude = [
    'subject_id', 'hadm_id', 'stay_id', 'icd_code', 'hospital_expire_flag',
    'diagnosis', 'gender', 'admittime', 'dischtime', 'deathtime', 'dob', 'dod',
    'dod_hosp', 'dod_ssn', 'stay_window', 'edregtime', 'edouttime'
]
X_raw = df.drop(columns=[col for col in cols_to_exclude if col in df.columns])

# =====================================================================
# 🔥 DIMENSIONALITY FIX: Limit One-Hot Encoding to Top 10 Diagnoses Only
# =====================================================================
print("Limiting feature space to top 10 most common clinical diagnoses...")
top_10_diagnoses = X_raw['icd_description'].value_counts().nlargest(10).index

# Group everything else into 'Other' to crush down those 2000+ sparse features
X_raw['icd_description_clean'] = X_raw['icd_description'].apply(lambda x: x if x in top_10_diagnoses else 'Other')

# Drop the raw un-collapsed description column before dummy transformation
X_raw = X_raw.drop(columns=['icd_description'])

# One-hot encode the heavily compacted column (results in only 11 clean columns total!)
X_encoded = pd.get_dummies(X_raw, columns=['icd_description_clean'], prefix='Dx', prefix_sep='_')
# =====================================================================

# 4. Clean up headers and handle any stray NaNs or text values
X_encoded = X_encoded.rename(columns=lambda x: re.sub(r'[^A-Za-z0-9_]+', '', x))

# Ensure every feature is completely numeric
stray_text = X_encoded.select_dtypes(exclude=[np.number]).columns
if len(stray_text) > 0:
    X_encoded = X_encoded.drop(columns=stray_text)

# Impute clinical vitals using median values to avoid mathematical fit gaps
for col in X_encoded.columns:
    if X_encoded[col].isnull().any():
        median_val = X_encoded[col].median()
        X_encoded[col] = X_encoded[col].fillna(median_val if not pd.isna(median_val) else 0)

# 5. Define continuous elements for standard scaling
continuous_cols = [
    'anchor_age', 'visit_procedure_count', 'vital_mean_heart_rate',
    'vital_mean_systolic_bp', 'vital_mean_diastolic_bp',
    'vital_mean_spo2', 'vital_mean_temp'
]
continuous_cols = [col for col in continuous_cols if col in X_encoded.columns]

# 6. GroupKFold Splitting
print("Executing patient-level GroupKFold split...")
gkf = GroupKFold(n_splits=5)
train_idx, test_idx = next(gkf.split(X_encoded, y, groups=groups))

X_train, X_test = X_encoded.iloc[train_idx].copy(), X_encoded.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train = groups.iloc[train_idx]

# 7. Normalize features
scaler = StandardScaler()
X_train[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])
X_test[continuous_cols] = scaler.transform(X_test[continuous_cols])

print(f"-> Tight Feature Matrix Established! Total Model Features: {X_train.shape[1]}")

# =====================================================================
# 8. Hyperparameter Search (RandomizedSearchCV over ElasticNet)
# =====================================================================
param_distributions = {
    'C': np.logspace(-2, 1, 10),
    'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
}

cv_strategy = GroupKFold(n_splits=5)

base_lr = LogisticRegression(
    penalty='elasticnet',
    solver='saga',
    class_weight='balanced',
    max_iter=500,  # Plenty of iterations for perfect convergence
    random_state=42
)

print("\nRunning fast RandomizedSearchCV over ElasticNet parameters...")
lr_search = RandomizedSearchCV(
    estimator=base_lr,
    param_distributions=param_distributions,
    n_iter=8,
    scoring='roc_auc',
    cv=cv_strategy.split(X_train, y_train, groups=groups_train),
    random_state=42,
    n_jobs=-1
)

lr_search.fit(X_train, y_train)

print("\n================ OPTIMIZATION SEARCH RESULTS COMPLETE ================")
print(f"Best Parameters Found -> C: {lr_search.best_params_['C']:.4f} | l1_ratio: {lr_search.best_params_['l1_ratio']:.2f}")

# 9. Final Test Assessment
best_model = lr_search.best_estimator_
y_prob_tuned = best_model.predict_proba(X_test)[:, 1]

test_summary = pd.DataFrame({
    'subject_id': df.iloc[test_idx]['subject_id'],
    'y_true': y_test,
    'y_prob': y_prob_tuned
})

# Compile down to strict patient-level metrics
patient_res_true = test_summary.groupby('subject_id')['y_true'].max()
patient_res_pred = test_summary.groupby('subject_id')['y_prob'].mean()
preds = (patient_res_pred >= 0.5).astype(int)

print("\n--- Optimized MIMIC-III Patient-Level Evaluation ---")
print(classification_report(patient_res_true, preds))
print(f"Optimized Patient-Level ROC-AUC Score: {roc_auc_score(patient_res_true, patient_res_pred):.4f}")

Mounted at /content/drive
Loading dataset...
Limiting feature space to top 10 most common clinical diagnoses...
Executing patient-level GroupKFold split...
-> Tight Feature Matrix Established! Total Model Features: 89

Running fast RandomizedSearchCV over ElasticNet parameters...

================ OPTIMIZATION SEARCH RESULTS COMPLETE ================
Best Parameters Found -> C: 0.0464 | l1_ratio: 0.70

--- Optimized MIMIC-III Patient-Level Evaluation ---
              precision    recall  f1-score   support

           0       0.97      1.00      0.99      4674
           1       1.00      0.87      0.93       970

    accuracy                           0.98      5644
   macro avg       0.99      0.94      0.96      5644
weighted avg       0.98      0.98      0.98      5644

Optimized Patient-Level ROC-AUC Score: 1.0000


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [14]:
import pandas as pd
import numpy as np
import re
from google.colab import drive
from sklearn.model_selection import GroupKFold, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score

# 1. Load the core dataset
drive.mount('/content/drive', force_remount=True)
base_path = '/content/drive/MyDrive/MIMIC_III_REAL/'

print("Loading dataset...")
df = pd.read_csv(base_path + 'MIMIC_III_CARDIAC_ML_READY.csv')

# 2. Extract targets and patient groups
y = df['hospital_expire_flag'].astype(int)
groups = df['subject_id']

# 3. Drop all raw tracking text, dates, and non-numeric administrative noise
cols_to_exclude = [
    'subject_id', 'hadm_id', 'stay_id', 'icd_code', 'hospital_expire_flag',
    'diagnosis', 'gender', 'admittime', 'dischtime', 'deathtime', 'dob', 'dod',
    'dod_hosp', 'dod_ssn', 'stay_window', 'edregtime', 'edouttime'
]
X_raw = df.drop(columns=[col for col in cols_to_exclude if col in df.columns])

# =====================================================================
# 🔥 FIX: Target Encoding to use 100% of diagnoses without splitting columns
# =====================================================================
print("Calculating risk-weight probabilities for ALL unique diagnosis strings...")
# Calculate the mean mortality rate for every single individual description string
diagnosis_risk_map = y.groupby(X_raw['icd_description']).mean()

# Map the global probabilities directly back onto the data column
X_raw['Dx_Encoded_Risk'] = X_raw['icd_description'].map(diagnosis_risk_map)

# Drop the raw description text column now that it's fully quantified
X_encoded = X_raw.drop(columns=['icd_description'])
# =====================================================================

# 4. Clean up headers and handle any stray NaNs or text values
X_encoded = X_encoded.rename(columns=lambda x: re.sub(r'[^A-Za-z0-9_]+', '', x))

# Ensure every feature is completely numeric
stray_text = X_encoded.select_dtypes(exclude=[np.number]).columns
if len(stray_text) > 0:
    X_encoded = X_encoded.drop(columns=stray_text)

# Impute clinical vitals using median values to avoid mathematical fit gaps
for col in X_encoded.columns:
    if X_encoded[col].isnull().any():
        median_val = X_encoded[col].median()
        X_encoded[col] = X_encoded[col].fillna(median_val if not pd.isna(median_val) else 0)

# 5. Define continuous elements for standard scaling (including our new risk score)
continuous_cols = [
    'anchor_age', 'visit_procedure_count', 'vital_mean_heart_rate',
    'vital_mean_systolic_bp', 'vital_mean_diastolic_bp',
    'vital_mean_spo2', 'vital_mean_temp', 'Dx_Encoded_Risk'
]
continuous_cols = [col for col in continuous_cols if col in X_encoded.columns]

# 6. GroupKFold Splitting
print("Executing patient-level GroupKFold split...")
gkf = GroupKFold(n_splits=5)
train_idx, test_idx = next(gkf.split(X_encoded, y, groups=groups))

X_train, X_test = X_encoded.iloc[train_idx].copy(), X_encoded.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train = groups.iloc[train_idx]

# 7. Normalize features
scaler = StandardScaler()
X_train[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])
X_test[continuous_cols] = scaler.transform(X_test[continuous_cols])

print(f"-> Feature Matrix Established! Total Model Features: {X_train.shape[1]}")

# =====================================================================
# 8. Hyperparameter Search (RandomizedSearchCV over ElasticNet)
# =====================================================================
param_distributions = {
    'C': np.logspace(-2, 1, 10),
    'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
}

cv_strategy = GroupKFold(n_splits=5)

base_lr = LogisticRegression(
    penalty='elasticnet',
    solver='saga',
    class_weight='balanced',
    max_iter=500,
    random_state=42
)

print("\nRunning fast RandomizedSearchCV over ElasticNet parameters...")
lr_search = RandomizedSearchCV(
    estimator=base_lr,
    param_distributions=param_distributions,
    n_iter=8,
    scoring='roc_auc',
    cv=cv_strategy.split(X_train, y_train, groups=groups_train),
    random_state=42,
    n_jobs=-1
)

lr_search.fit(X_train, y_train)

print("\n================ OPTIMIZATION SEARCH RESULTS COMPLETE ================")
print(f"Best Parameters Found -> C: {lr_search.best_params_['C']:.4f} | l1_ratio: {lr_search.best_params_['l1_ratio']:.2f}")

# 9. Final Test Assessment
best_model = lr_search.best_estimator_
y_prob_tuned = best_model.predict_proba(X_test)[:, 1]

test_summary = pd.DataFrame({
    'subject_id': df.iloc[test_idx]['subject_id'],
    'y_true': y_test,
    'y_prob': y_prob_tuned
})

# Compile down to strict patient-level metrics
patient_res_true = test_summary.groupby('subject_id')['y_true'].max()
patient_res_pred = test_summary.groupby('subject_id')['y_prob'].mean()
preds = (patient_res_pred >= 0.5).astype(int)

print("\n--- Optimized MIMIC-III Patient-Level Evaluation ---")
print(classification_report(patient_res_true, preds))
print(f"Optimized Patient-Level ROC-AUC Score: {roc_auc_score(patient_res_true, patient_res_pred):.4f}")

Mounted at /content/drive
Loading dataset...
Calculating risk-weight probabilities for ALL unique diagnosis strings...
Executing patient-level GroupKFold split...
-> Feature Matrix Established! Total Model Features: 90

Running fast RandomizedSearchCV over ElasticNet parameters...

================ OPTIMIZATION SEARCH RESULTS COMPLETE ================
Best Parameters Found -> C: 0.0464 | l1_ratio: 0.70

--- Optimized MIMIC-III Patient-Level Evaluation ---
              precision    recall  f1-score   support

           0       0.97      1.00      0.99      4674
           1       1.00      0.87      0.93       970

    accuracy                           0.98      5644
   macro avg       0.99      0.94      0.96      5644
weighted avg       0.98      0.98      0.98      5644

Optimized Patient-Level ROC-AUC Score: 1.0000


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
